In [7]:
# !pip cache purge
# borra el caché local de paquetes de pip

Files removed: 0


In [ ]:
# ==============================================================================
# FASE 1: Configuración del entorno de ejecución
# ==============================================================================

# Instalación silenciosa ('-q') y actualización ('-U') de dependencias críticas.
# Se requiere 'transformers' >= 4.40.0 para soporte nativo de la arquitectura Llama 3.
# 'accelerate' permite la carga inteligente del modelo con 'device_map="auto"'.
# 'bitsandbytes' es el motor de cuantización para reducir la precisión a 4-bits.
# 'huggingface_hub' gestiona la autenticación y descarga desde repositorios "gated".

!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

'''
Notas técnicas:
1. 'transformers': Proporciona las abstracciones de alto nivel para el modelo y el tokenizador.
2. 'accelerate': Abstrae la complejidad de mover tensores entre CPU y GPU, optimizando el uso de la VRAM disponible.
3. 'bitsandbytes': Implementa la cuantización "NormalFloat 4" (NF4), vital para correr 8B parámetros en la T4 de Colab.
4. 'huggingface_hub': Necesaria para validar el "HF_TOKEN" y acceder a modelos con licencia de Meta.
'''

In [ ]:
# ==============================================================================
# FASE 2: Importación de módulos y gestión de identidad (Secretos)
# ==============================================================================

import torch  # Motor de tensores y gestión de memoria en GPU (VRAM).
from transformers import (
    AutoTokenizer,          # Abstracción para la tokenización automática según el modelo.
    AutoModelForCausalLM,   # Clase para cargar modelos generativos (Causal Language Modeling).
    BitsAndBytesConfig      # Configuración de cuantización para optimización de hardware.
)
from google.colab import userdata  # Interfaz para acceder a variables de entorno seguras.
import json  # Manejo de la serialización y deserialización de las respuestas del modelo.

'''
Gestión de Seguridad:
Se utiliza 'userdata' para inyectar el token de Hugging Face desde el entorno
de ejecución de Colab. Esto previene la exposición accidental de credenciales
en el historial de Git (Hardcoded Credentials) al subir el notebook a un repo público.
'''
hf_token = userdata.get('HF_TOKEN')

'''
Propósito Técnico:
1. 'torch' se utiliza aquí principalmente para definir los tipos de datos (dtype)
   durante la inferencia (e.g., bfloat16).
2. Las clases 'Auto*' permiten que el código sea agnóstico al modelo;
   cambiar Llama 3 por otro LLM solo requiere modificar el 'model_id'.'''

In [ ]:
# ==============================================================================
# FASE 3: Carga y Optimización de Memoria (Quantization & Loading)
# ==============================================================================

# Se utiliza la versión de 'unsloth' que ya integra pesos pre-cuantizados en 4-bits.
# Esto reduce el 'payload' de descarga de ~16GB a ~5.7GB, permitiendo operar
# dentro de los límites de la RAM de sistema de la versión gratuita de Colab.
model_id = 'unsloth/llama-3-8b-Instruct-bnb-4bit'

# 'BitsAndBytesConfig' define la estrategia de compresión de los pesos.
# 1. 'load_in_4bit': Activa la cuantización de 4 bits.
# 2. 'bnb_4bit_use_double_quant': Cuantiza también las constantes de cuantización (ahorro extra de memoria).
# 3. 'bnb_4bit_quant_type': 'nf4' (NormalFloat 4) es un tipo de dato óptimo para pesos con distribución normal.
# 4. 'bnb_4bit_compute_dtype': El cálculo interno se hace en 'bfloat16' para mayor estabilidad y velocidad en GPUs T4/A100.
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype = torch.bfloat16
)

print('Instanciando Tokenizador y Model Engine...')

# El tokenizador es esencial para convertir strings en tensores de entrada compatibles con Llama 3.
tokenizer = AutoTokenizer.from_pretrained(model_id, token = hf_token)

'''
Parámetros críticos de carga:
- 'device_map': "auto" utiliza la librería 'accelerate' para calcular automáticamente
  la distribución de las capas del modelo en los dispositivos disponibles (GPU/CPU).
- 'low_cpu_mem_usage': Evita cargar el modelo completo en RAM de sistema antes de
  moverlo a la GPU, previniendo el fallo del kernel por 'Out of Memory' (OOM).
'''
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config = bnb_config,
    device_map = 'auto',
    low_cpu_mem_usage = True,
    token = hf_token
)

print('Carga completada: Modelo mapeado exitosamente en VRAM.')

In [ ]:
# Llama 3 no tiene pad_token por defecto. Usamos el EOS como pad.
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
#     model.config.pad_token_id = tokenizer.eos_token_id

# print(f"Pad token configurado: {tokenizer.pad_token}")

In [ ]:
# ==============================================================================
# FASE 4: Definición del Pipeline de Inferencia (ABSA Logic)
# ==============================================================================

def analizar_sentimiento_absa(texto):
    '''
    Ejecuta un análisis de sentimiento basado en aspectos (ABSA) utilizando Llama 3.

    El proceso sigue el flujo: Prompt Engineering -> Tokenización de Plantilla ->
    Inferencia Determinista -> Post-procesamiento de Tensores.
    '''

    # 1. Definición del System Prompt (Context Steering):
    # Se utiliza una técnica de "Strict JSON Enforcement" para asegurar que el
    # output sea parseable programáticamente sin preámbulos innecesarios.
    system_prompt = """Eres un experto en NLP y análisis de sentimiento multidimensional.
Tu tarea es analizar el texto proporcionado y extraer las entidades o aspectos clave mencionados, evaluando el sentimiento específico hacia cada uno.
DEBES responder ÚNICAMENTE con un objeto JSON válido. No incluyas saludos, explicaciones ni texto fuera del JSON.
El JSON debe tener esta estructura:
{
  "analisis": [
    {
      "aspecto": "Nombre de la entidad o tema",
      "sentimiento": "Positivo | Negativo | Neutro",
      "justificacion": "Breve explicación basada en el texto"
    }
  ]
}"""

    # 2. Estructuración del mensaje (Chat Markup):
    # Llama 3 utiliza etiquetas especiales (<|start_header_id|>, etc.) para separar roles.
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Analiza el siguiente texto:\n\n{texto}"}
    ]

    # 3. Tokenización y Formateo de Plantilla:
    # 'apply_chat_template' renderiza el diccionario 'messages' al formato de texto
    # plano que el modelo espera ver según su entrenamiento instructivo.
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True, # Añade el header <|start_header_id|>assistant para iniciar la respuesta.
        return_tensors="pt",        # Retorna tensores de PyTorch.
        return_dict=True            # Incluye input_ids y attention_mask.
    ).to(model.device)              # Desplazamiento de tensores a la memoria de la GPU (VRAM).

    # 4. Configuración de Terminadores de Secuencia:
    # Es vital incluir '<|eot_id|>' (End Of Turn) para que el modelo no siga
    # generando texto aleatorio una vez que ha cerrado el objeto JSON.
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    # 5. Generación del Output (Inferencia):
    # Usamos 'do_sample=False' para forzar una "Greedy Search". Esto garantiza que el
    # modelo elija siempre el token más probable, reduciendo la variabilidad (alucinaciones)
    # en la estructura del JSON.
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        eos_token_id=terminators,
        pad_token_id=tokenizer.pad_token_id, # Evita errores de puntero nulo (NoneType).
        do_sample=False,
    )

    # 6. Post-procesamiento de la Salida (Tensor Slicing):
    # El modelo devuelve los tokens de entrada concatenados con los de salida.
    # Obtenemos solo los tokens nuevos cortando el tensor desde el final del input.
    input_length = inputs['input_ids'].shape[-1]
    response_tokens = outputs[0][input_length:]

    # 7. Decodificación Final:
    # Convierte los IDs numéricos de vuelta a texto legible, omitiendo tokens especiales.
    return tokenizer.decode(response_tokens, skip_special_tokens=True)

In [ ]:
# ==============================================================================
# FASE 5: Ejecución de Test y Validación de Estructura de Datos
# ==============================================================================

# Definición de un caso de prueba con 'polaridad mixta'.
# Este tipo de texto es el 'benchmark' ideal para ABSA, ya que contiene
# sentimientos positivos y negativos entrelazados que un modelo de
# clasificación global no podría segmentar correctamente.
texto_prueba = 'El reciente torneo de Taekwondo en la provincia tuvo un nivel organizativo impecable, atrayendo a competidores de gran nivel de toda la región. Sin embargo, el costo de las inscripciones fue excesivamente alto, lo que generó bastante malestar en las academias locales debido a la falta de apoyo de sponsors.'

print('Iniciando inferencia en GPU (Warmup)...')

# Invocación de la lógica de análisis definida en la Fase 4.
resultado_json_str = analizar_sentimiento_absa(texto_prueba)

'''
Lógica de Resiliencia y Parsing:
Los LLMs, aunque potentes, son modelos probabilísticos que pueden fallar en
la sintaxis del formato solicitado. Implementamos un bloque 'try-except'
para manejar errores de decodificación, una práctica obligatoria en
desarrollo de nivel 'middle' para evitar la ruptura del pipeline.
'''
try:
    # Deserialización: Convertimos el string de salida en un objeto (diccionario) de Python.
    resultado_dict = json.loads(resultado_json_str)

    print('\n¡Éxito! Salida estructurada recuperada:')

    # 'json.dumps' con 'indent' se utiliza para realizar un "Pretty Print" del JSON,
    # facilitando la lectura humana y la depuración de los aspectos extraídos.
    # Usamos double quotes internas para cumplir con el estándar JSON.
    print(json.dumps(resultado_dict, indent=2, ensure_ascii=False))

except json.JSONDecodeError:
    '''
    Manejo de Excepciones:
    Si Llama 3 devuelve texto adicional (alucinación de formato) o un JSON
    mal formado, capturamos el error para analizar la salida cruda (Raw Output)
    y ajustar el System Prompt si fuera necesario.
    '''
    print('\nError: El modelo falló al generar un formato JSON válido.')
    print(f'Salida cruda del modelo:\n{resultado_json_str}')

In [ ]:
# ==============================================================================
# FASE 6: Configuración del Baseline (pysentimiento / BETO)
# ==============================================================================

# Instalación de 'pysentimiento'. Esta librería es un 'wrapper' optimizado que
# utiliza modelos basados en BETO (Spanish BERT). Es el estándar de facto para
# tareas de clasificación rápida de texto en español debido a su balance entre
# precisión y costo computacional.
!pip install -q pysentimiento

from pysentimiento import create_analyzer

'''
Arquitectura del Baseline:
A diferencia de Llama 3 (que es un modelo 'Decoder-only' de 8B parámetros),
pysentimiento utiliza un modelo tipo 'Encoder' (BETO).
Esto lo hace órdenes de magnitud más ligero y especializado exclusivamente
en entender la semántica de las etiquetas de sentimiento ('POS', 'NEG', 'NEU')
sin la carga de la generación de texto.
'''

# Instanciamos el analizador:
# - 'task': Definimos "sentiment" para clasificación de polaridad.
# - 'lang': "es" asegura el uso del modelo entrenado con el corpus de la RAE y redes sociales.
analyzer = create_analyzer(task = "sentiment", lang = "es")

print('Baseline (pysentimiento) cargado: Motor listo para el benchmark.')

In [ ]:
# ==============================================================================
# FASE 7: Ejecución del Benchmark y Normalización de Salidas
# ==============================================================================

import pandas as pd

# Definición del Corpus de Prueba:
# Se seleccionan textos con 'polaridad mixta' y contextos regionales (Misiones/Chaco).
# El objetivo es forzar la divergencia entre el modelo de clasificación global
# (Baseline) y el modelo de extracción de aspectos (Llama 3).
data_test = [
    'El torneo de Taekwondo en Posadas tuvo una organización impecable, pero el precio del buffet era un robo.',
    'La nueva librería de Python para LLMs es increíblemente rápida, aunque la documentación es escasa.',
    'El clima en Misiones hoy es un desastre, no para de llover y hay mucha humedad.',
    'Me encantó el curso de IA, aprendí mucho sobre Transformers y redes neuronales.',
    'La gestión de sponsors para el viaje al Chaco es frustrante, nadie responde los correos.'
]

resultados_comparativos = []

print(f'Iniciando procesamiento por lotes: {len(data_test)} muestras encontradas.')

for texto in data_test:
    # 1. Inferencia con Baseline (pysentimiento):
    # Este modelo devuelve un objeto de clase 'SentimentOutput'.
    # Extraemos el atributo '.output' que contiene las etiquetas estándar: 'POS', 'NEG' o 'NEU'.
    res_base = analyzer.predict(texto)
    sentimiento_base = res_base.output

    # 2. Inferencia con Llama 3 (Función ABSA personalizada):
    # El LLM devuelve un string con formato JSON.
    res_llama_raw = analizar_sentimiento_absa(texto)

    try:
        # Deserialización del output del LLM:
        res_llama_dict = json.loads(res_llama_raw)

        '''
        Lógica de Alineación de Datos:
        Dado que Llama 3 realiza un análisis por aspectos (múltiples sentimientos),
        y el Baseline solo ofrece uno global, tomamos el primer aspecto extraído
        como el "sentimiento predominante" para permitir una comparación directa.
        Normalizamos el texto a mayúsculas y truncamos a los primeros 3 caracteres
        para igualar el formato 'POS', 'NEG', 'NEU' del baseline.
        '''
        sentimiento_llama = res_llama_dict["analisis"][0]["sentimiento"].upper()[:3]
    except Exception as e:
        # Manejo de excepciones en caso de que el LLM devuelva un JSON mal formado.
        sentimiento_llama = 'ERR'

    # Almacenamiento estructurado para posterior análisis con Pandas:
    resultados_comparativos.append({
        'texto': texto[:50] + '...', # Truncado para mejorar la legibilidad de la tabla final.
        'baseline': sentimiento_base,
        'llama3': sentimiento_llama
    })

# Construcción del DataFrame de métricas:
# Transformamos la lista de diccionarios en una estructura tabular, facilitando
# el cálculo de precisión y la generación de matrices de confusión en la siguiente fase.
df_metricas = pd.DataFrame(resultados_comparativos)

print('\nAnálisis finalizado. Resultados consolidados en DataFrame:')
print(df_metricas)

In [ ]:
# ==============================================================================
# FASE 8: Visualización de Métricas y Matriz de Divergencia
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

'''
Preparación de Datos (Data Normalization):
Para que los gráficos sean legibles por humanos (Stakeholders), mapeamos los
códigos técnicos ('POS', 'NEG', 'NEU') a etiquetas descriptivas.
'''
mapping = {'POS': "Positivo", 'NEG': "Negativo", 'NEU': "Neutro"}
df_plot = df_metricas.copy()

# Aplicamos el mapeo al baseline (BETO)
df_plot['baseline'] = df_plot['baseline'].map(mapping)

# Normalizamos la salida de Llama 3 para asegurar compatibilidad en la matriz.
# Buscamos la presencia de los prefijos para asignar la categoría final.
df_plot['llama3'] = df_plot['llama3'].apply(
    lambda x: "Positivo" if "POS" in x else ("Negativo" if "NEG" in x else "Neutro")
)

'''
1. Cálculo de Coincidencia (Agreement Score):
Dado que no tenemos un "Ground Truth" (etiquetado humano), lo que medimos aquí
técnicamente es la tasa de acuerdo entre el modelo especializado y el LLM.
'''
accuracy = accuracy_score(df_plot['baseline'], df_plot['llama3'])

# 2. Configuración de la Visualización
plt.figure(figsize = (12, 5))

'''
Gráfico de Barras (Sentiment Distribution):
Transformamos el DataFrame a formato 'long-form' usando 'melt' para que Seaborn
pueda comparar las frecuencias de ambos modelos en un solo eje.
'''
plt.subplot(1, 2, 1)
df_melted = df_plot.melt(value_vars = ['baseline', 'llama3'], var_name = 'Modelo', value_name = 'Sentimiento')
sns.countplot(data = df_melted, x = 'Sentimiento', hue = 'Modelo', palette = 'viridis')
plt.title(f'Distribución de Sentimientos (Coincidencia: {accuracy:.2%})')

'''
Matriz de Divergencia (Confusion Matrix):
Visualiza dónde discrepan los modelos. Un valor fuera de la diagonal principal
indica un caso donde el LLM 'vio' algo distinto al modelo baseline, lo cual
es oro puro para el debugging de prompts o análisis de sarcasmo.
'''
plt.subplot(1, 2, 2)
cm = confusion_matrix(
    df_plot['baseline'],
    df_plot['llama3'],
    labels = ['Positivo', "Neutro", "Negativo"]
)
sns.heatmap(
    cm, annot = True, fmt = 'd', cmap = 'Blues',
    xticklabels = ['Positivo', "Neutro", "Negativo"],
    yticklabels = ['Positivo', "Neutro", "Negativo"]
)
plt.title('Matriz de Divergencia (Baseline vs Llama 3)')
plt.ylabel('Baseline (BETO)')
plt.xlabel('Llama 3 (Generativo)')

plt.tight_layout()
plt.show()